# OMM Object

The OMM (i.e. Orbit Mean-Elements Message, as defined by the CCSDS Recommended Standard 502.0-B-3) is the format Space-Track distributes together with the TLEs. It carries the very same SGP4 mean elements, but without the constraints of the two fixed-width lines: this makes it possible to represent objects whose catalog number does not fit the TLE format, and to store the elements at their full precision.

In this notebook we discuss the usage of the OMM object: this allows the user to create an OMM object from a dictionary, a string or a file, in any of the four serializations of the standard (JSON, XML, KVN and CSV). Since an OMM object is a `dsgp4.tle.TLE` object under the hood, it can be used everywhere a TLE is expected (for a tutorial on the propagation see: [TLE propagation](tle_propagation.ipynb)).

## Imports

In [ ]:
import dsgp4
import torch

## Load OMM from `dict`

Here, we show how to load an OMM from a dictionary of OMM fields: this is, for instance, what the Space-Track API returns when the JSON format is requested:

In [ ]:
omm_fields = {
    "CCSDS_OMM_VERS": "3.0",
    "CREATION_DATE": "2022-02-28T09:16:12",
    "ORIGINATOR": "18 SPCS",
    "OBJECT_NAME": "SENTINEL-1A",
    "OBJECT_ID": "2014-016A",
    "CENTER_NAME": "EARTH",
    "REF_FRAME": "TEME",
    "TIME_SYSTEM": "UTC",
    "MEAN_ELEMENT_THEORY": "SGP4",
    "EPOCH": "2022-02-28T01:57:54.918432",
    "MEAN_MOTION": "14.59199732",
    "ECCENTRICITY": "0.0001341",
    "INCLINATION": "98.1819",
    "RA_OF_ASC_NODE": "68.1874",
    "ARG_OF_PERICENTER": "82.4703",
    "MEAN_ANOMALY": "277.6657",
    "EPHEMERIS_TYPE": "0",
    "CLASSIFICATION_TYPE": "U",
    "NORAD_CAT_ID": "39634",
    "ELEMENT_SET_NO": "999",
    "REV_AT_EPOCH": "42107",
    "BSTAR": "0.000021846",
    "MEAN_MOTION_DOT": "0.00000057",
    "MEAN_MOTION_DDOT": "0",
}

#let us construct the OMM object
omm = dsgp4.omm.OMM(omm_fields)
print(omm)

All right, we can now access the elements as if it was a dictionary, or as attributes of the class, exactly as with a TLE object (note that the elements are stored in SI units, while the OMM fields are in degrees and rev/day):

In [ ]:
print("OMM elements:")
print(f"Object name: {omm.name}")
print(f"Satellite catalog number: {omm.satellite_catalog_number}")
print(f"International designator: {omm.international_designator}")
print(f"Epoch year: {omm.epoch_year}")
print(f"Epoch day: {omm.epoch_days}")
print(f"Epoch (MJD): {omm.date_mjd}")
print(f"Inclination [rad]: {omm._inclo}")
print(f"Right ascension of the ascending node [rad]: {omm._nodeo}")
print(f"Eccentricity [-]: {omm._ecco}")
print(f"Argument of perigee [rad]: {omm._argpo}")
print(f"Mean anomaly [rad]: {omm._mo}")
print(f"Mean motion [rad/min]: {omm._no_kozai}")
print(f"BSTAR drag term: {omm._bstar}")

#the raw OMM fields, as they were read, remain available:
print(f"\nOriginator: {omm._fields['ORIGINATOR']}, mean motion [rev/day]: {omm._fields['MEAN_MOTION']}")

The handy methods of the TLE object are available here too:

In [ ]:
#let's first define the Earth radius according to WSG-84:
r_earth=dsgp4.util.get_gravity_constants('wgs-84')[2].numpy()*1e3

print(f"Semi-major axis [km]: {omm.semi_major_axis*1e-3}")
print(f"Apogee altitude [km]: {omm.apogee_alt(r_earth)*1e-3}")
print(f"Perigee altitude [km]: {omm.perigee_alt(r_earth)*1e-3}")

## Load OMM from `str`

The standard defines four serializations: JSON, XML, KVN (i.e. keyword-value notation) and CSV. `dsgp4` reads and writes all of them, and detects the format automatically:

In [ ]:
kvn = dsgp4.omm.dumps(omm, file_format='kvn')
print(kvn)

#the format does not have to be specified, it is detected from the content:
print(f"Detected format: {dsgp4.omm.detect_format(kvn)}")
omm_from_kvn = dsgp4.omm.OMM(kvn)
print(f"Same mean motion: {omm_from_kvn._no_kozai == omm._no_kozai}")

In [ ]:
#the same holds for the other three serializations:
for file_format in ['json', 'xml', 'csv']:
    text = dsgp4.omm.dumps(omm, file_format=file_format)
    other = dsgp4.omm.OMM(text)
    print(f"{file_format}: {len(text)} characters, satellite catalog number {other.satellite_catalog_number}")

## Load OMMs from file

Here, we load the OMMs directly from file. We assume the user has downloaded the OMM data, for instance from [Space-Track](https://www.space-track.org/), and has placed such data in the `example_omm.json` file.

Then, we can simply load all OMMs using `dsgp4`. This will construct a list of OMM objects:

```{note}
The format is detected from the file extension and content, but it can also be passed explicitly via the `file_format` argument.
```

In [ ]:
omms = dsgp4.omm.load('example_omm.json')
print(f"Loaded {len(omms)} OMMs: {[o.name for o in omms]}")
print(f"Catalog numbers: {[o.satellite_catalog_number for o in omms]}")

## From OMM to TLE, and back

Since the two formats carry the same elements, an OMM object can be converted into a TLE object and vice versa:

In [ ]:
tle = omm.to_tle()
print(tle)

#and the other way around:
print(tle.to_omm()._fields['EPOCH'])

There is one thing the OMM format can do that the TLE format cannot: the two lines only have five characters for the satellite catalog number, so, even with the Alpha-5 convention, they cannot represent objects numbered above 339999. Such objects are only distributed as OMMs:

In [ ]:
big_catalog_number = dsgp4.omm.OMM(dict(omm_fields, NORAD_CAT_ID='700123'))
print(f"Satellite catalog number: {big_catalog_number.satellite_catalog_number}")

try:
    big_catalog_number.to_tle()
except ValueError as e:
    print(f"As expected, this object cannot be written as a TLE:\n{e}")

## Propagation

Finally, OMM objects are propagated exactly like TLE objects, both one at a time and in batches:

In [ ]:
#we initialize the propagator, and propagate at 0, 10 and 100 minutes since the OMM epoch:
dsgp4.initialize_tle(omm)
states = dsgp4.propagate(omm, torch.tensor([0., 10., 100.]))
print(f"Position [km]:\n{states[:, 0, :]}")
print(f"Velocity [km/s]:\n{states[:, 1, :]}")

#and, in batch, over the OMMs we loaded from file:
batch_states = dsgp4.propagate_batch(omms, torch.tensor([0., 10., 100.]), initialized=False)
print(f"\nBatch of states: {batch_states.shape}")

Since the OMM object is a TLE object, everything else in `dsgp4` (e.g. the partial derivatives, the covariance transformations, or the ML-dSGP4 model) works with OMMs without any change.